# 01 - Data exploration

**Goal:** Understand the source data before choosing cleaning rules.

The CSV stays local. The preview uses selected operational fields and excludes names, emails, and free text. The remaining checks report aggregate results.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project = next(
    folder for folder in (cwd, *cwd.parents)
    if (folder / "src" / "dataset_audit.py").is_file()
)
csv_path = project.parent / "tickets.csv"

# Keep blank fields missing and preserve text such as a literal "NA".
df = pd.read_csv(csv_path, keep_default_na=False, na_values=[""])
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
preview_columns = ["created_at", "category", "product", "channel", "priority", "status"]
display(df[preview_columns].head())
df.info()

Rows: 100,000 | Columns: 38


,created_at,category,product,channel,priority,status
0,2023-01-12 16:51:00,Authentication & Access,Payment Gateway,Phone,Low,Closed
1,2024-09-03 15:58:00,Integrations & API,Desktop Client,Self-Service,Low,Closed
2,2023-06-15 09:50:00,Authentication & Access,Desktop Client,Email,Low,Open
3,2022-12-15 09:57:00,Subscription,API Platform,Chat,Medium,Closed
4,2024-01-10 08:09:00,Integrations & API,Mobile App,Email,Low,Closed


<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 38 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ticket_id                    100000 non-null  int64  
 1   created_at                   100000 non-null  str    
 2   issue_description            100000 non-null  str    
 3   category                     100000 non-null  str    
 4   issue_type_id                100000 non-null  str    
 5   product                      100000 non-null  str    
 6   channel                      100000 non-null  str    
 7   region                       100000 non-null  str    
 8   language                     100000 non-null  str    
 9   customer_name                100000 non-null  str    
 10  customer_email               100000 non-null  str    
 11  customer_age                 100000 non-null  int64  
 12  customer_gender              100000 non-null  str    
 13  subscriptio

## Types and completeness

`shape` gives dimensions, `head()` shows the first records, and `info()` shows pandas' inferred dtypes and non-null counts. Dates stored as strings still need explicit parsing. `isna().sum()` counts missing values and `nunique()` counts distinct non-missing values. Compare those counts to spot IDs, low-cardinality categories, and sparse fields.

In [2]:
# Count whitespace-only text as missing without changing the source dataframe.
observed = df.replace(r"^\s*$", pd.NA, regex=True)
missing = observed.isna().sum()
profile = pd.DataFrame({
    "inferred_dtype": df.dtypes.astype(str),
    "missing": missing,
    "missing_pct": (100 * missing / len(df)).round(2),
    "distinct_nonmissing": observed.nunique(),
})
display(profile.sort_values("missing", ascending=False, kind="stable"))
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate nonmissing IDs: {observed['ticket_id'].dropna().duplicated().sum():,}")

,inferred_dtype,missing,missing_pct,distinct_nonmissing
escalation_reason,str,72505,72.50,3
customer_satisfaction_score,float64,10146,10.15,5
escalated,str,10068,10.07,2
resolution_time_hours,float64,10068,10.07,7500
resolution_wallclock_hours,float64,10068,10.07,20998
first_resolved_at,str,10068,10.07,82984
resolved_at,str,10068,10.07,83357
resolved_by,str,10068,10.07,35
sla_breach_margin,float64,10068,10.07,12086
closed_at,str,6809,6.81,90160


Exact duplicate rows: 0
Duplicate nonmissing IDs: 0


## Numeric distributions

`describe()` summarizes each numeric column. Compare the median and 95th percentile with the maximum to see how far the tail extends. Exclude the ticket ID because its magnitude has no operational meaning. Extreme values are candidates for investigation and can be legitimate.

In [3]:
numeric = df.select_dtypes(include="number").drop(columns="ticket_id", errors="ignore")
display(numeric.describe(percentiles=[.25, .5, .75, .95, .99]).T.round(2))
display(pd.crosstab(
    observed["status"],
    observed["resolution_wallclock_hours"].notna().map({True: "Recorded", False: "Missing"}),
))

,count,mean,std,min,25%,50%,75%,95%,99%,max
customer_age,100000.0,45.96,16.45,18.00,32.00,46.00,60.00,72.00,74.00,74.00
customer_tenure_months,100000.0,35.46,20.78,0.00,18.00,35.00,53.00,68.00,71.00,71.00
previous_tickets,100000.0,2.99,1.73,0.00,2.00,3.00,4.00,6.00,8.00,12.00
issue_complexity_score,100000.0,4.68,2.50,1.00,3.00,5.00,7.00,9.00,10.00,10.00
reopen_count,100000.0,0.31,0.99,0.00,0.00,0.00,0.00,3.00,4.00,8.00
first_response_time_hours,100000.0,3.11,6.96,0.02,0.18,0.45,2.40,16.43,35.70,133.37
resolution_time_hours,89932.0,13.35,63.12,-85.16,0.68,1.78,10.34,72.51,98.75,7160.78
paused_hours,100000.0,20.77,148.10,0.00,0.00,0.00,5.40,61.14,219.65,2877.94
resolution_wallclock_hours,89932.0,106.40,230.41,0.05,7.63,29.33,105.55,421.99,949.77,10025.09
sla_target_hours,100000.0,65.07,41.93,4.00,24.00,72.00,120.00,120.00,120.00,120.00


resolution_wallclock_hours,Missing,Recorded
status,,
Cancelled,3689,0
Closed,0,89502
New,47,0
On-Hold,706,0
Open,4574,0
Pending,1052,0
Resolved,0,430


## Timestamp and duration consistency

`errors="coerce"` converts unparseable timestamps to missing values in the temporary date table so we can count parse failures. Subtracting creation from final resolution gives a separate elapsed-time estimate. Compare it with the recorded wall-clock duration using a tolerance of 0.01 hours (36 seconds). This is a diagnostic tolerance for rounding and timestamp precision.

In [4]:
date_columns = ["created_at", "first_resolved_at", "resolved_at", "closed_at"]
dates = observed[date_columns].apply(pd.to_datetime, format="mixed", errors="coerce")
display(pd.DataFrame({
    "parsed": dates.notna().sum(),
    "nonblank_unparseable": (observed[date_columns].notna() & dates.isna()).sum(),
    "earliest": dates.min(),
    "latest": dates.max(),
}))

elapsed_hours = (dates["resolved_at"] - dates["created_at"]).dt.total_seconds() / 3600
wallclock = pd.to_numeric(observed["resolution_wallclock_hours"], errors="coerce")
comparable = elapsed_hours.notna() & wallclock.notna()
gap = wallclock.sub(elapsed_hours).abs()

checks = pd.Series({
    "First resolution before creation": dates["first_resolved_at"].lt(dates["created_at"]).sum(),
    "Final resolution before creation": elapsed_hours.lt(0).sum(),
    "Final resolution before first resolution": dates["resolved_at"].lt(dates["first_resolved_at"]).sum(),
    "Closure before final resolution": dates["closed_at"].lt(dates["resolved_at"]).sum(),
    "Comparable wall-clock records": comparable.sum(),
    "Wall-clock differs from timestamp elapsed by > 0.01 h": (comparable & gap.gt(.01)).sum(),
    "Negative recorded resolution_time_hours": df["resolution_time_hours"].lt(0).sum(),
}, name="tickets")
display(checks.to_frame())

,parsed,nonblank_unparseable,earliest,latest
created_at,100000,0,2022-01-01 09:16:00,2024-12-30 23:21:00
first_resolved_at,89932,0,2022-01-02 23:18:00,2025-01-16 17:05:00
resolved_at,89932,0,2022-01-02 23:18:00,2025-02-05 10:03:00
closed_at,93191,0,2022-01-03 02:17:00,2025-01-03 06:27:00


,tickets
First resolution before creation,284
Final resolution before creation,124
Final resolution before first resolution,456
Closure before final resolution,284
Comparable wall-clock records,89932
Wall-clock differs from timestamp elapsed by > 0.01 h,1548
Negative recorded resolution_time_hours,111


## Findings and questions for cleaning

- The CSV has 100,000 rows and 38 columns. Pandas inferred 25 string and 13 numeric columns. `ticket_id` is a record key. The date strings become usable timestamps after parsing.
- The four date columns had no unparseable nonblank values. Event order still needs review. There are 124 final resolutions before creation, 456 final resolutions before first resolution, and 284 closures before final resolution. These checks can flag the same ticket.
- Among 89,932 tickets with both duration measures available for comparison (recorded wall-clock and timestamp-derived elapsed), 1,548 differ by more than 0.01 hours. Inspect the discrepancies before choosing which duration to use for the label.
- The median recorded wall-clock time is 29.33 hours, the 95th percentile is 421.99 hours, and the maximum is 10,025.09 hours. The long tail warrants investigation before any exclusions.
- All 10,068 missing wall-clock values occur outside Closed and Resolved statuses. Keep this distinction when defining the population for analysis and modeling.